# Stage 2 Classification Calibration Audit

**Objective:** verify that the Stage 2 experimental proposal preserves immutable protections while reducing unresolved cards to approximately 10% of all 1,726 L4 cards.

**Authority boundary:** all new Stage 2 placements are algorithmic proposals, not human-approved ground truth. No charts or images are produced.

In [1]:
from pathlib import Path
import json
from collections import Counter

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parents[1]
STAGE2 = ROOT / 'data/experiments/stage2-v1'
VALIDATION = ROOT / 'reports/validation/stage2-v1'

def load(path):
    return json.loads(path.read_text(encoding='utf-8'))

placements = load(STAGE2 / 'stage2_placements.json')
summary = load(STAGE2 / 'stage2_summary.json')
config = load(STAGE2 / 'stage2_config.json')
validation = load(VALIDATION / 'validation_summary.json')
print({'rows': len(placements), 'stage2_id': summary['stage2_id']})

{'rows': 1726, 'stage2_id': 'stage2-v1'}


## Planned checks

1. Grain and identity remain one row per RAI4 ID.
2. Physical locks and Stage 1 consensus are unchanged.
3. Unresolved count is 173 (about 10%).
4. New placements retain review flags and do not claim human approval.
5. Concentration and high-risk override segments are quantified.

In [2]:
ids = [row['l4_id'] for row in placements]
status_counts = Counter(row['stage2_status'] for row in placements)
checks = {
    'rows_1726': len(placements) == 1726,
    'unique_ids_1726': len(set(ids)) == 1726,
    'physical_locked_182': status_counts['locked_physical'] == 182,
    'stage1_consensus_37': status_counts['stage1_algorithm_proposed'] == 37,
    'unresolved_173': status_counts['needs_taxonomy_decision'] == 173,
    'no_human_approval_claim': not any(row['human_approved'] for row in placements),
    'legacy_hierarchy_excluded': not any(row['legacy_hierarchy_used_as_feature'] for row in placements),
}
print(status_counts)
print(checks)
assert all(checks.values())

Counter({'algorithm_proposed_stage2': 1334, 'locked_physical': 182, 'needs_taxonomy_decision': 173, 'stage1_algorithm_proposed': 37})
{'rows_1726': True, 'unique_ids_1726': True, 'physical_locked_182': True, 'stage1_consensus_37': True, 'unresolved_173': True, 'no_human_approval_claim': True, 'legacy_hierarchy_excluded': True}


In [3]:
new_rows = [row for row in placements if row['stage2_status'] == 'algorithm_proposed_stage2']
holds = [row for row in placements if row['stage2_status'] == 'needs_taxonomy_decision']
quality_profile = {
    'new_stage2_assignments': len(new_rows),
    'unresolved': len(holds),
    'unresolved_rate': round(len(holds) / len(placements), 6),
    'without_rule_support': sum(not row['rule_supported'] for row in new_rows),
    'taxonomy_gap_overrides': sum(row['taxonomy_gap_override'] for row in new_rows),
    'low_fit_tier': sum(row['stage2_fit_tier'] == 'stage2_low' for row in new_rows),
}
print(quality_profile)

{'new_stage2_assignments': 1334, 'unresolved': 173, 'unresolved_rate': 0.100232, 'without_rule_support': 1214, 'taxonomy_gap_overrides': 247, 'low_fit_tier': 653}


In [4]:
assigned_counts = Counter(row['stage2_l3_id'] for row in placements if row['stage2_l3_id'])
top_five = assigned_counts.most_common(5)
print({'top_five_l3': top_five, 'largest_share': summary['largest_l3_share_of_all_assigned']})
assert summary['largest_l3_share_of_all_assigned'] < 0.20

{'top_five_l3': [('RAI3-G-INT-10', 228), ('RAI3-G-SYS-08', 146), ('RAI3-G-SYS-03', 104), ('RAI3-G-INT-08', 100), ('RAI3-G-INT-06', 81)], 'largest_share': 0.146813}


In [5]:
soft_holds = sorted(
    row['stage2_suitability_score'] for row in holds
    if row['stage2_hold_reason'] == 'BOTTOM_10_PERCENT_SUITABILITY_RESERVE'
)
soft_assignments = sorted(
    row['stage2_suitability_score'] for row in new_rows
    if row['stage2_suitability_score'] is not None
)
print({
    'last_soft_hold': max(soft_holds),
    'first_soft_assignment': min(soft_assignments),
    'hard_hold_count': summary['hard_hold_count'],
    'quota_hold_count': summary['quota_hold_count'],
})

{'last_soft_hold': 0.250476, 'first_soft_assignment': 0.251094, 'hard_hold_count': 55, 'quota_hold_count': 118}


In [6]:
print({
    'validation_status': validation['status'],
    'passed_checks': validation['passed_checks'],
    'failed_checks': validation['failed_checks'],
    'warnings': validation['warning_count'],
})
assert validation['status'] == 'PASS_WITH_WARNINGS'
assert validation['failed_checks'] == 0

{'validation_status': 'PASS_WITH_WARNINGS', 'passed_checks': 13, 'failed_checks': 0, 'warnings': 3}


## Result and next step

Stage 2 meets its operational target: 1,553 of 1,726 cards are placed and 173 remain unresolved. Physical locks and Stage 1 consensus are unchanged. The result is intentionally more permissive: rule-free and taxonomy-gap override assignments require priority human review. Stage 3 may force-match the remaining 173 only if its outputs preserve the original hold reason and are visibly labeled as forced, uncalibrated, and not human-approved.